# Pandas (2)

__Авторы задач: Блохин Н.В. (NVBlokhin@fa.ru), Макрушин С.В. (SVMakrushin@fa.ru)__

Материалы:
* Макрушин С.В. "Лекция 2: Библиотека Pandas"
* https://pandas.pydata.org/docs/user_guide/index.html#
* https://pandas.pydata.org/docs/reference/index.html
* Уэс Маккини. Python и анализ данных

## Задачи для совместного разбора

1. Загрузите данные из файла `sp500hst.txt` и обозначьте столбцы в соответствии с содержимым: `"date", "ticker", "open", "high", "low", "close", "volume"`.

2. Посчитайте количество уникальных цифр, которые используются каждой строке в столбце volume.

3. Для каждой строки рассчитайте разность между значениями high и low, если индекс столбца нечетный, и разность между close и high в противном случае.

4. Рассчитайте суммарный объем торгов для для одинаковых значений тикеров.

5. Загрузите данные из файла sp500hst.txt и обозначьте столбцы в соответствии с содержимым: "date", "ticker", "open", "high", "low", "close", "volume". Добавьте столбец с расшифровкой названия тикера, используя данные из файла `sp_data2.csv` . В случае нехватки данных об именах тикеров корректно обработать их.

## Лабораторная работа №2.2

__Данная работа является продолжением ЛР №2. Для начала работы загрузите таблицы (см. задание 1.1)__

In [2]:
import pandas as pd

In [28]:
recipes = pd.read_csv("02_pandas_data/recipes_sample.csv", parse_dates = ['submitted'] )
reviews = pd.read_csv("02_pandas_data/reviews_sample.csv", parse_dates= ['date'], index_col=0)

In [11]:
recipes.head()

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN


In [12]:
reviews.head()

,user_id,recipe_id,date,rating,review
370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...


### Применение функций к pd.Series и pd.DataFrame

4.1 Напишите функцию, которая переводит минуты в формат "XhYm". Примените эту функцию к столбцу `minutes` таблицы `recipes` (без перезаписи исходного столбца) при помощи метода `map`.

In [21]:
def m_to_hm(m: int) -> str:
    """Convert minutes to hours and minutes"""
    return f"{m // 60}h{m % 60}m"

assert m_to_hm(90) == "1h30m"
assert m_to_hm(10) == "0h10m"

In [22]:
recipes["minutes"].map(m_to_hm)

0        1h30m
1        0h10m
2        0h30m
3        0h45m
4        0h25m
         ...  
29995    1h20m
29996     4h0m
29997    1h15m
29998     1h0m
29999    0h29m
Name: minutes, Length: 30000, dtype: object

4.2 На основе таблицы `recipes` создайте таблицу, которая содержит только текстовые столбцы (используйте метод `select_dtypes`).  Примените к каждому элементу этой таблицы строковый метод `str.capitalize` при помощи метода `applymap`, не удаляя пропуски.

In [23]:
recipes_new = recipes.select_dtypes(include="object") # or exclude="number"
recipes_new.applymap(lambda x: x.capitalize() if isinstance(x, str) else x) 
recipes_new.head()

,name,description
0,george s at the cove black bean soup,an original recipe created by chef scott meska...
1,healthy for them yogurt popsicles,my children and their friends ask for my homem...
2,i can t believe it s spinach,"these were so go, it surprised even me."
3,italian gut busters,my sister-in-law made these for us at a family...
4,love is in the air beef fondue sauces,i think a fondue is a very romantic casual din...


4.3 Напишите функцию, которая принимает на вход серию `pd.Series` и для серий, содержащих текстовые данные, возвращает максимальную длину строк в ней, а для числовых серий возвращает минимальный элемент в этой серии. Примените данную функцию к каждому столбцу таблицы `recipes` при помощи метода `apply`.

In [24]:
def get_stats(x: pd.Series) -> int:
    """Get max len for str and min for int"""
    if x.dtype == "object":
        return x.str.len().max()
    else:
        return x.min()

assert get_stats(pd.Series(['a', 'bbbb', 'ccc'])) == 4
assert get_stats(pd.Series([1, 3, 2])) == 1

In [25]:
recipes = recipes.apply(get_stats)

In [26]:
recipes.head()

name                               83
id                                 48
minutes                             0
contributor_id                   1530
submitted         1999-08-06 00:00:00
dtype: object

### Группировки таблиц `pd.DataFrame`

5.1 Посчитайте количество рецептов, представленных каждым из участников (`contributor_id`). Какой участник добавил максимальное кол-во рецептов?

In [29]:
recipes_count = recipes.groupby("contributor_id").size()
recipes_count

contributor_id
1530            5
1533          186
1534           50
1535           40
1538            8
             ... 
2001968497      2
2002059754      1
2002234079      1
2002234259      1
2002247884      1
Length: 8404, dtype: int64

In [32]:
recipes_count.idxmax()

89831

5.2 Посчитайте средний рейтинг к каждому из рецептов. Для скольких рецептов отсутствуют отзывы? Обратите внимание, что отзыв с нулевым рейтингом или не заполненным текстовым описанием не считается отсутствующим.

In [36]:
recipes_mean = reviews.groupby("recipe_id")["rating"].mean()
recipes_mean

recipe_id
48        1.000000
55        4.750000
66        4.944444
91        4.750000
94        5.000000
            ...   
536547    5.000000
536610    0.000000
536728    4.000000
536729    4.750000
536747    0.000000
Name: rating, Length: 28100, dtype: float64

5.3 Посчитайте количество рецептов с разбивкой по годам создания.

In [37]:
recipes_years = recipes.groupby(recipes["submitted"].dt.year).size()
recipes_years

submitted
1999     275
2000     104
2001     589
2002    2644
2003    2334
2004    2153
2005    3130
2006    3473
2007    4429
2008    4029
2009    2963
2010    1538
2011     922
2012     659
2013     490
2014     139
2015      42
2016      24
2017      39
2018      24
dtype: int64

5.4 Напишите функцию, которая принимает на вход таблицу (аналогичную `recipes` по набору столбцов), и возвращает `True` в том случае, если в столбце `minutes` присутствуют только значения, меньшие либо равные 10. Сгруппируйте таблицу `recipes` по полю `contributor_id` и для каждого участника выясните, справедливо ли, что все его рецепты занимают не более 10 минут.

In [39]:
import numpy as np

In [40]:
def has_only_fast_recipes(x: pd.DataFrame) -> bool:
    """Check if all recipes are fast"""
    return x["minutes"].max() <= 10

assert not has_only_fast_recipes(
    pd.DataFrame(
        {
            "name": {0: "george s", 1: "healthy"},
            "id": {0: 44123, 1: 67664},
            "minutes": {0: 90, 1: 10},
            "contributor_id": {0: 35193, 1: 91970},
            "submitted": {0: "2002-10-25", 1: "2003-07-26"},
            "n_steps": {0: np.nan, 1: np.nan},
            "description": {0: "123", 1: "zxc"},
            "n_ingredients": {0: 18.0, 1: np.nan},
        }
    )
)
assert has_only_fast_recipes(
    pd.DataFrame(
        {
            "name": {0: "george s", 1: "healthy"},
            "id": {0: 44123, 1: 67664},
            "minutes": {0: 7, 1: 5},
            "contributor_id": {0: 35193, 1: 91970},
            "submitted": {0: "2002-10-25", 1: "2003-07-26"},
            "n_steps": {0: np.nan, 1: np.nan},
            "description": {0: "123", 1: "zxc"},
            "n_ingredients": {0: 18.0, 1: np.nan},
        }
    )
)

In [41]:
recipes.groupby("contributor_id").apply(has_only_fast_recipes)

contributor_id
1530          False
1533          False
1534          False
1535          False
1538          False
              ...  
2001968497    False
2002059754    False
2002234079    False
2002234259    False
2002247884    False
Length: 8404, dtype: bool

In [42]:
if "True" in recipes.groupby("contributor_id").apply(has_only_fast_recipes).astype(str).values:
    print("Not all recipes are fast")

Not all recipes are fast


### Объединение таблиц `pd.DataFrame`

6.1 При помощи объединения таблиц, создайте `DataFrame`, состоящий из четырех столбцов: `id`, `name`, `user_id`, `rating`. Рецепты, на которые не оставлен ни один отзыв, должны отсутствовать в полученной таблице. Подтвердите правильность работы вашего кода, выбрав рецепт, не имеющий отзывов, и попытавшись найти строку, соответствующую этому рецепту, в полученном `DataFrame`.

In [61]:
new_df = recipes.merge(reviews, left_on="id", right_on="recipe_id", how="left")
new_df = new_df[["id", "name", "user_id", "rating"]]
new_df = new_df.dropna(subset=["rating"])
new_df

,id,name,user_id,rating
0,44123,george s at the cove black bean soup,743566.0,5.0
1,44123,george s at the cove black bean soup,76503.0,5.0
2,44123,george s at the cove black bean soup,34206.0,5.0
3,67664,healthy for them yogurt popsicles,494084.0,5.0
4,67664,healthy for them yogurt popsicles,303445.0,5.0
...,...,...,...,...
128591,486161,zydeco soup,305531.0,5.0
128592,486161,zydeco soup,1271506.0,5.0
128593,486161,zydeco soup,724631.0,5.0
128594,486161,zydeco soup,133174.0,5.0


6.2 При помощи объединения таблиц и группировок, создайте `DataFrame`, состоящий из трех столбцов: `recipe_id`, `name`, `review_count`, где столбец `review_count` содержит кол-во отзывов, оставленных на рецепт `recipe_id`. У рецептов, на которые не оставлен ни один отзыв, в столбце `review_count` должен быть указан 0. Подтвердите правильность работы вашего кода, выбрав рецепт, не имеющий отзывов, и найдя строку, соответствующую этому рецепту, в полученном `DataFrame`.

In [62]:
df_new = recipes.merge(reviews, left_on="id", right_on="recipe_id", how="left")
df_new = df_new.groupby(["recipe_id", "name"]).size().reset_index(name="review_count")
# если у рецепта нет отзывов, то в столбце review_count будет NaN, поэтому заменим его на 0
df_new["review_count"] = df_new["review_count"].fillna(0)
df_new

,recipe_id,name,review_count
0,48.0,boston cream pie,2
1,55.0,betty crocker s southwestern guacamole dip,4
2,66.0,black coffee barbecue sauce,18
3,91.0,brown rice and vegetable pilaf,4
4,94.0,blueberry buttertarts,4
...,...,...,...
28095,536547.0,cauliflower ceviche,1
28096,536610.0,miracle home made puff pastry,1
28097,536728.0,gluten free vegemite,1
28098,536729.0,creole watermelon feta salad,4


6.3. Выясните, рецепты, добавленные в каком году, имеют наименьший средний рейтинг?

In [63]:
reviews.groupby(recipes["submitted"].dt.year)["rating"].mean().idxmin()

2000.0

### Сохранение таблиц `pd.DataFrame`

7.1 Отсортируйте результат выполнения задания 6.1 в порядке убывания величины столбца `id` и сохраните результаты в csv файл. 

In [64]:
new_df.sort_values(by="id", ascending=False).to_csv("new_df.csv", index=False)

7.2 Воспользовавшись `pd.ExcelWriter`, cохраните результаты 6.1 и 6.2 в файл: на лист с названием `Рецепты с оценками` сохраните результаты выполнения 6.1; на лист с названием `Количество отзывов по рецептам` сохраните результаты выполнения 6.2.

In [65]:
with pd.ExcelWriter("new_df.xlsx") as writer:
    new_df.to_excel(writer, sheet_name="Рецепты с оценками", index=False)
    df_new.to_excel(writer, sheet_name="Количество отзывов по рецептам", index=False)
